# Tutorial 01 — The `eunomia.profiles/v1` Profile DSL & `alexandriac` Compiler

In a sovereign dataspace, semantic and validation artifacts (`vocab.ttl`, `context.jsonld`, `shapes.ttl`, `schema.json`, `cedar.schema`, `prof.ttl`, `spec.html`, and `space-lock.json`) are **never edited by hand**. The dataspace authority maintains a single declarative source of truth in YAML using the **`eunomia.profiles/v1`** DSL, and the **`alexandriac`** compiler validates all 5 normative levels (`N1`–`N5`) and deterministically generates every artifact.

### What You Will Learn in This Notebook
1. Build and run the `bin/alexandriac` CLI compiler.
2. Inspect and compile an ODRL policy profile (`kind: OdrlProfile`).
3. Build a complete dataspace bundle (`OdrlProfile` + `SpaceProfile`) and inspect the generated `space-lock.json` and self-contained `schema.json` (JSON Schema Draft 2020-12).
4. Trigger and diagnose validation errors across levels **N1, N2, N3, and N4** (including the `active -> deprecated -> retired` governance lifecycle rule).

In [1]:
import json
import subprocess
import tempfile
from pathlib import Path
from lab import ensure_binaries, repo_root, show_artifacts, show_yaml

ROOT = repo_root()
BIN_DIR = ensure_binaries()
ALEXANDRIAC = BIN_DIR / "alexandriac"
WORK = Path(tempfile.mkdtemp(prefix="alexandriac-tutorial-"))
print(f"Repository root: {ROOT}")
print(f"Compiler binary: {ALEXANDRIAC}")

Repository root: /Users/apabook/Desktop/alexandria
Compiler binary: /Users/apabook/Desktop/alexandria/bin/alexandriac


## 1. Validate and Compile an `OdrlProfile` (`colors.profile.yaml`)

A `kind: OdrlProfile` document defines:
- Profile metadata (`id`, `version`, `title`, `publisher`, `license`, `base_url`).
- RDF prefixes (`namespaces`).
- Adopted ODRL 2.2 terms (`adopted.actions`, `adopted.left_operands`).
- Custom profile terms (`action`, `left_operand` with `signature` and optional Cedar `enforcement` mapping, and `right_operand`).

Let's run `alexandriac validate` and `alexandriac compile` against `internal/dsl/testdata/colors.profile.yaml`:

In [2]:
colors_yaml = ROOT / "internal/dsl/testdata/colors.profile.yaml"
show_yaml("colors.profile.yaml", colors_yaml.read_text(encoding="utf-8"), subtitle="ODRL profile with Cedar enforcement mapping (PDP-4)")

res_val = subprocess.run([str(ALEXANDRIAC), "validate", str(colors_yaml)], capture_output=True, text=True)
print("✔ alexandriac validate:", (res_val.stdout or res_val.stderr).strip())

out_colors = WORK / "colors-out"
res_cmp = subprocess.run([str(ALEXANDRIAC), "compile", "-o", str(out_colors), str(colors_yaml)], capture_output=True, text=True)

✔ alexandriac validate: ok: 1 documents valid


### Interactive Inspection of Generated Artifacts
Notice how from a single 66-line YAML file, `alexandriac` deterministically generated **8 artifacts**, including `cedar-mapping.json` and `cedar.schema` (because the `color` operand declared an `enforcement` block):

In [3]:
show_artifacts("Compiled artifacts for colors/0.1.0", out_colors / "profiles/odrl/colors/0.1.0")

## 2. Create and Compile a Full Dataspace Bundle (`OdrlProfile` + `SpaceProfile`)

On the Vocabulary Hub (`alexandria-server`), each dataspace draft is a **YAML stream** (documents separated by `---`) containing the `SpaceProfile` and the internal `OdrlProfile` documents referenced by its `bindings`.

Let's author version `1.0.0` of a mobility dataspace (`https://example.com/spaces/mobility`) and compile it:

In [4]:
bundle_v1 = WORK / "mobility-v1.0.0.yaml"
bundle_v1.write_text('''dsl: eunomia.profiles/v1
kind: OdrlProfile
profile:
  id: https://example.com/odrl/profile/mobility
  version: 1.0.0
  title: { en: Mobility Data Usage Profile }
  publisher: GING-DIT, Universidad Politécnica de Madrid
  license: https://creativecommons.org/licenses/by/4.0/
  base_url: https://example.com/odrl/profile/mobility
namespaces:
  mob: https://example.com/ns/mobility#
adopted:
  left_operands:
    - term: odrl:dateTime
      signature:
        datatype: xsd:dateTime
        operators: [eq, lt, lteq, gt, gteq]
terms:
  - id: aggregateTelemetry
    kind: action
    included_in: odrl:use
    label: { en: Aggregate Telemetry }
    definition: { en: Compute anonymized fleet telemetry aggregates. }
  - id: minFleetSize
    kind: left_operand
    label: { en: Minimum Fleet Size }
    definition: { en: Minimum number of vehicles in an aggregation window. }
    signature:
      datatype: xsd:integer
      operators: [gt, gteq]
---
dsl: eunomia.profiles/v1
kind: SpaceProfile
space:
  id: https://example.com/spaces/mobility
  version: 1.0.0
  title: { en: Mobility Sovereign Dataspace }
  operator: https://example.com/party/mobility-authority
  base_url: https://example.com/spaces/mobility
lifecycle:
  grace_window: P30D
  propagation_window: P2D
  clock_skew: PT5S
  lease_margin: PT10S
  max_concurrent_versions: 2
  freshness:
    ttl: PT24H
    max_staleness: P7D
  degraded:
    policy_intake: continue
    dsp_message: continue
bindings:
  - slot: odrl
    profile: https://example.com/odrl/profile/mobility
    source: internal
    version: "=1.0.0"
    obligation: mandatory
    conformance:
      - { role: json-schema, applies_to: policy_intake }
      - { role: shacl, applies_to: policy_intake }
''', encoding="utf-8")

out_mob_v1 = WORK / "mobility-v1-out"
res = subprocess.run([str(ALEXANDRIAC), "compile", "-o", str(out_mob_v1), str(bundle_v1)], capture_output=True, text=True, check=True)
show_yaml("mobility-v1.0.0.yaml (OdrlProfile + SpaceProfile Bundle)", bundle_v1.read_text(encoding="utf-8"), subtitle="Complete dataspace YAML stream")
show_artifacts("Compiled Bundle (Mobility Space 1.0.0)", out_mob_v1)

## 3. Testing Validation Levels `N1`–`N4`

The compiler verifies 5 validation levels before emitting any package:
- **N1:** Strict JSON-over-YAML model (no duplicate keys or anchors) + normative JSON Schema validation with line numbers.
- **N2:** Semantic invariants, alias collisions, operator–datatype compatibility matrix (§9.4), and `LIFECYCLE_*` invariants (§2.10).
- **N3:** Adopted terms checked against the closed ODRL 2.2 vocabulary and SemVer range resolution (`=`, `^`, `~`).
- **N4:** Cross-version diff against `-previous`: forbids deleting terms (`TERM_REMOVED`) and requires transitioning through `deprecated` before `retired` (`TERM_RETIRED_WITHOUT_DEPRECATION`).

Let's first trigger an **N4** error by attempting to retire `aggregateTelemetry` directly in `2.0.0` without deprecating it first, and then validate the proper gradual transition (`1.0.0 -> 1.1.0 deprecated -> 2.0.0 retired`):

In [5]:
# Invalid N4 case: transition directly from active (1.0.0) to retired (2.0.0) without deprecation.
bad_v2 = WORK / "mobility-bad-v2.yaml"
bad_v2.write_text(
    bundle_v1.read_text(encoding="utf-8")
    .replace("version: 1.0.0", "version: 2.0.0")
    .replace('"=1.0.0"', '"=2.0.0"')
    .replace("kind: action", "kind: action\n    status: retired\n    replaced_by: odrl:use"),
    encoding="utf-8",
)

res_bad = subprocess.run(
    [str(ALEXANDRIAC), "validate", "-previous", str(bundle_v1), str(bad_v2)],
    capture_output=True,
    text=True,
)
print("❌ Expected N4 error when retiring a term without prior deprecation:")
print("Exit code:", res_bad.returncode)
print(res_bad.stderr.strip())

# Valid N4 evolution: includes intermediate 1.1.0 (deprecated) and 2.0.0 (retired).
good_v2 = ROOT / "conformance/f6/drafts/v2.0.0.yaml"
good_v1 = ROOT / "conformance/f6/drafts/v1.0.0.yaml"
res_good = subprocess.run(
    [str(ALEXANDRIAC), "validate", "-previous", str(good_v1), str(good_v2)],
    capture_output=True,
    text=True,
)
print("\n✅ Valid gradual transition 1.0.0 -> 1.1.0 (deprecated) -> 2.0.0 (retired):")
print("Exit code:", res_good.returncode)
print(res_good.stdout.strip())

❌ Expected N4 error when retiring a term without prior deprecation:
Exit code: 1
error   TERM_RETIRED_WITHOUT_DEPRECATION   mobility-bad-v2.yaml#1:21 /terms/0/status mob:aggregateTelemetry must be deprecated in a previous release before it is retired
alexandriac: 1 errors

✅ Valid gradual transition 1.0.0 -> 1.1.0 (deprecated) -> 2.0.0 (retired):
Exit code: 0
ok: 3 documents valid
